# EduCrew Studio: free AI episode maker
**Series:** EduCrew: From a Little Dream to a Bigger World (10 episodes, 3D animated, 9:16)
Every step uses free, open-source models on Kaggle's free GPU, so there are no per-video charges.

**Before you start (one time):**
1. In Kaggle's right panel, open **Session options**: set **Accelerator** to **GPU T4 x2** and turn **Internet** on. Kaggle asks for phone verification the first time.
2. Create a free Hugging Face account. Accept the terms on https://huggingface.co/ai4bharat/indic-parler-tts, then create a *Read* token at https://huggingface.co/settings/tokens.
3. In Kaggle, open **Add-ons → Secrets**, add the token with the label `HF_TOKEN`, and tick it so it's attached.
4. Optional: upload your logo as a Kaggle dataset with the file named `educrew_logo.png`, and attach it with **Add Input**.

Then run the cells from top to bottom. Video clips are slow on free GPUs, so you can do a few shots per session and run **Save progress** at the end.

**Order for the series:**
1. Run once with `EPISODE = "episodes/EduCrew_Characters.json"`, but only the **Setup** and **Step 1** cells. This makes the character bible images that keep her the same person in every episode. Check them, and redo any you don't like.
2. Then make episodes **in order**, 01 to 10, because each episode starts from the previous episode's last image.

In [ ]:
import glob, os, shutil
# ▶ WHICH VIDEO TO MAKE: put the name of any file from the episodes/ folder here
EPISODE = "episodes/EduCrew_Biography_Episode_01.json"   # or episodes/EduCrew_Characters.json

PROJECT = "/kaggle/working/educrew-studio"
if not os.path.exists(PROJECT):
    !git clone -q https://github.com/educatorscircleec-dot/educrew-studio {PROJECT}
else:
    !cd {PROJECT} && git pull -q   # fetch any new videos added to GitHub
os.chdir(PROJECT)
NAME = os.path.splitext(os.path.basename(EPISODE))[0]
WORK = f"work/{NAME}"
print("Making:", NAME, "| available:", sorted(os.listdir("episodes")))
# Logo: copy it from an attached Kaggle dataset, if there is one
for p in glob.glob("/kaggle/input/**/educrew_logo.png", recursive=True):
    shutil.copy(p, "logo/educrew_logo.png"); print("logo found:", p)
# Resume: if you attached a saved work.zip from an earlier session, unpack it
for z in glob.glob("/kaggle/input/**/work.zip", recursive=True):
    shutil.unpack_archive(z, PROJECT); print("restored", z)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
!pip install -q -U diffusers transformers accelerate ftfy imageio-ffmpeg sentencepiece compel

## Step 1: keyframes (one 3D storybook still per shot)
Look at the results. To redo a shot, add `--shots 3 --force --seed 55` to the command.

In [ ]:
!cd tools && python make_keyframes.py ../$EPISODE
from IPython.display import display, Image as Im
for p in sorted(glob.glob(f"{WORK}/keyframes/*.png")): print(p); display(Im(p, width=220))

## Step 2: video clips (slowest step)
Run a few shots at a time. Finished clips are skipped automatically. Shots without a clip still work: the assembler animates the still image instead.
If you still run out of memory, add `--offload sequential` (slower, uses much less GPU memory).

In [ ]:
!cd tools && python make_clips.py ../$EPISODE --shots 1,2,3

## Step 3: voices (narrator, mother, child), exact words from the episode file

In [ ]:
!python -m venv /tmp/tts && /tmp/tts/bin/pip install -q torch soundfile transformers sentencepiece git+https://github.com/huggingface/parler-tts.git
!cd tools && /tmp/tts/bin/python make_voices.py ../$EPISODE
from IPython.display import Audio
for p in sorted(glob.glob(f"{WORK}/voice/*.wav")): print(p); display(Audio(p))

## Step 4: realistic sound effects (no music)

In [ ]:
if not os.path.exists("/kaggle/working/MMAudio"):
    !git clone -q https://github.com/hkchengrex/MMAudio /kaggle/working/MMAudio
    !pip install -q -e /kaggle/working/MMAudio
!cd tools && python make_sfx.py ../$EPISODE --mmaudio /kaggle/working/MMAudio

## Step 5: assemble the final 60-second Reel with the logo

In [ ]:
!python assemble.py $EPISODE
from IPython.display import Video
Video(glob.glob(f"output/*{NAME}.mp4")[0], embed=True, width=300)

## Save progress
Download `work.zip` (and the final video from `output/`) from the **Output** panel. Next session, upload `work.zip` as a dataset to continue where you left off.

In [ ]:
shutil.make_archive("/kaggle/working/work", "zip", PROJECT, "work")
print("saved /kaggle/working/work.zip")